In [5]:
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from selenium.webdriver.common.action_chains import ActionChains

import undetected_chromedriver as uc # IMPORTANTE: Usar esta librería

import time
import random
import pandas as pd

In [ ]:
# Links
meli_countries_home_links = {
    "Argentina": "https://www.mercadolibre.com.ar/",
    "Bolivia": "https://www.mercadolibre.com.bo/",
    "Brasil": "https://www.mercadolivre.com.br/",
    "Chile": "https://www.mercadolibre.cl",
    "Colombia": "https://www.mercadolibre.com.co/",
    "Costa Rica": "https://www.mercadolibre.com.cr/",
    "Ecuador": "https://www.mercadolibre.com.ec/",
    "El Salvador": "https://www.mercadolibre.com.sv/",
    "Honduras": "https://www.mercadolibre.com.hn/",
    "Guatemala": "https://www.mercadolibre.com.gt/",
    "Paraguay": "https://www.mercadolibre.com.py/",
    "Uruguay": "https://www.mercadolibre.com.uy/",
    "México": "https://www.mercadolibre.com.mx/",
    "Panamá": "https://www.mercadolibre.com.pa/",
    "Honduras": "https://www.mercadolibre.com.hn/",
    "Perú": "https://www.mercadolibre.com.pe/",
    "República Dominicana": "https://www.mercadolibre.com.do/",
    "Venezuela": "https://www.mercadolibre.com.ve/"
}

meli_countries_cellphones = {
    "Argentina": "https://listado.mercadolibre.com.ar/celulares",
    "Bolivia": "https://listado.mercadolibre.com.bo/celulares",
    #"Brasil": "https://lista.mercadolivre.com.br/celulares",
    "Brasil": "https://lista.mercadolivre.com.br/celulares-telefones/celulares-smartphones/celulares_Desde_1489_NoIndex_True",
    "Chile": "https://listado.mercadolibre.cl/celulares",
    "Costa Rica": "https://listado.mercadolibre.co.cr/celulares",
    #"Colombia": "https://listado.mercadolibre.com.co/celulares",
    "Colombia": "https://listado.mercadolibre.com.co/celulares-telefonos/celulares-smartphones/celulares_Desde_1441_NoIndex_True",
    "Ecuador": "https://listado.mercadolibre.com.ec/celulares",
    "Ecuador": "https://listado.mercadolibre.com.ec/celulares-y-telefonia/celulares-y-smartphones/celulares_Desde_1105_NoIndex_True",
    "El Salvador": "https://listado.mercadolibre.com.sv/celulares",
    "Honduras": "https://listado.mercadolibre.com.hn/celulares",
    "Guatemala": "https://listado.mercadolibre.com.gt/celulares",
    # "México": "https://listado.mercadolibre.com.mx/celulares",
    "México": "https://listado.mercadolibre.com.mx/celulares-telefonia/celulares-smartphones/celulares_Desde_433_NoIndex_True",
    "Panamá": "https://listado.mercadolibre.com.pa/celulares",
    "Paraguay": "https://listado.mercadolibre.com.py/celulares",
    "Uruguay": "https://listado.mercadolibre.com.uy/celulares",
    "Honduras": "https://listado.mercadolibre.com.hn/celulares",
    "Perú": "https://listado.mercadolibre.com.pe/celulares",
    "República Dominicana": "https://listado.mercadolibre.com.do/celulares",
    #"Venezuela": "https://listado.mercadolibre.com.ve/celulares"
    "Venezuela": "https://listado.mercadolibre.com.ve/celulares-y-telefonos/celulares-y-smartphones/celulares_Desde_1009_NoIndex_True"

}

In [7]:
def iniciar_driver():
    options = uc.ChromeOptions()
    #options.add_argument("--headless")
    options.add_argument("--disable-cache")
    options.add_argument("--disk-cache-size=0")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1920,5000")
    options.add_argument("--disable-gpu")

    options.add_argument("--disable-extensions")

    try:
        # Iniciamos con undetected_chromedriver
        # No necesitas pasar el 'service', UC lo gestiona solo
        driver = uc.Chrome(options=options, headless=False, version_main=146) 
        
        # UC ya inyecta el parche para navigator.webdriver, 
        # pero podemos reforzarlo si quieres:
        driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
            "source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
        })
        
        return driver
    except Exception as e:
        print(f"❌ Error crítico al iniciar: {e}")
        return None

def recolectar_links_por_prefijo(driver):
    links_validos = set()
    
    class_name = "poly-component__title"

    print(f"🔍 Buscando específicamente clases que inicien con: {class_name}")

    driver.save_screenshot("debug_paso_0.png")

    # Realizamos un barrido con scroll para capturar los dinámicos
    for i in range(8):
        driver.execute_script("window.scrollBy(0, 700);")

        driver.save_screenshot(f"debug_paso_{i}.png")

        # Buscamos TODOS los enlaces <a> en la página
        todos_los_a = driver.find_elements(By.CLASS_NAME, class_name)
        
        conteo_antes = len(links_validos)
        for el in todos_los_a:
            try:
                href = el.get_attribute("href")
                # Verificamos si el link existe
                if href:
                    links_validos.add(href)
            except:
                continue
        
        nuevos = len(links_validos) - conteo_antes
        print(f"   📥 Scroll {i+1}: Encontrados {nuevos} nuevos (Total: {len(links_validos)})")

        time.sleep(random.uniform(2,4))

        # Si ya tenemos un buen número, podemos seguir
        if len(links_validos) >= 60:
            break

    return list(links_validos)


def scrapear_mercadolibre(country, n_page):

    driver = iniciar_driver()
    
    wait = WebDriverWait(driver, 120) # Aumentamos el tiempo de espera por si acaso
    tiny_wait = WebDriverWait(driver, 30)
    resultados = []

    next_button_flag = 1

    print("🏠 Entrando a la Home para 'calentar' la sesión...")
    driver.get(meli_countries_home_links[country])
    time.sleep(random.uniform(4, 8))

    driver.execute_script(f"window.scrollTo(0, {random.uniform(300, 700)});")

    page_link = meli_countries_cellphones[country]

    while next_button_flag:
        
        print(f"🚀 Accediendo a: {page_link}")
        driver.get(page_link)
        wait.until(EC.url_to_be(page_link))

        try:            
            # 1. Esperar a que los elementos "Poly" carguen
            # Usamos un selector CSS que busque la clase que mencionaste
            selector_enlace = 'a.poly-component__title'

            try:
                print("🔎 Buscando enlaces de productos...")
                wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, selector_enlace)))

                cards = driver.find_elements(By.CLASS_NAME, "poly-card")
                print(f"Total de tarjetas detectadas: {len(cards)}")

                # Usamos un set para evitar links duplicados (a veces el título y la imagen tienen el mismo link)
                urls = recolectar_links_por_prefijo(driver)
                
                print(f"✅ Se encontraron {len(urls)} productos únicos.")
            except Exception as e:
                print("Problemas detección poly title: ", e)
                
            else:
                for i, url in enumerate(urls): 
                    print(f"--- [{i+1}/{len(urls)}] Navegando a producto ---")
                    
                    if i + 1 % 8 == 0:
                        time.sleep(random.uniform(60, 80))
                    try:
                        driver.get(url)
                    except:
                        continue

                    # Delay humano para que cargue la ficha técnica
                    time.sleep(random.uniform(15, 20))

                    try:
                        # Extraer Datos con los nuevos selectores de la página de producto
                        nombre = wait.until(EC.presence_of_element_located((By.CLASS_NAME, "ui-pdp-title"))).text
                        precio = driver.find_element(By.CSS_SELECTOR, "span.andes-money-amount__fraction").text
                        moneda = driver.find_element(By.CSS_SELECTOR, "span.andes-money-amount__currency-symbol").text
                        
                        # Vendedor
                        try:
                            vendedor = driver.find_element(By.CSS_SELECTOR, "a.ui-pdp-media__action.ui-pdp-seller__link").text
                        except:
                            vendedor = "No detectado"

                        # Especificaciones (Tabla dinámica)
                        driver.execute_script("window.scrollTo(0, 800);")

                        actions = ActionChains(driver)
                        
                        try:
                            boton_specs = tiny_wait.until(EC.element_to_be_clickable((By.ID, "see-more-button-hs-features")))
                            driver.execute_script("arguments[0].scrollIntoView(true);", boton_specs)

                            actions.move_to_element(boton_specs).perform() # Mueve el cursor al elemento

                            driver.execute_script("arguments[0].click();", boton_specs)

                        except Exception as e:
                            print("   ℹ️ El botón no estaba presente (quizás las specs ya están visibles).")
                            print(e)

                        try:
                            boton_specs = wait.until(EC.element_to_be_clickable((By.CLASS_NAME, "ui-pdp-collapsable__action.ui-vpp-highlighted-specs__striped-collapsed__action")))
                            driver.execute_script("arguments[0].scrollIntoView(true);", boton_specs)

                            time.sleep(random.uniform(2,4))

                            # actions = ActionChains(driver)
                            actions.move_to_element(boton_specs).perform() # Mueve el cursor al elemento

                            driver.execute_script("arguments[0].click();", boton_specs)

                            print("   ✅ Tabla expandida con éxito.")
                        except Exception as e:
                            print("No hay specs")
                            print(e)

                        specs = {}
                        filas_xpath = "//tr[contains(@class, 'andes-table__row')]"
                        filas = driver.find_elements(By.XPATH, filas_xpath)

                        for fila in filas:
                            try:
                                key = fila.find_element(By.XPATH, ".//th").text
                                val = fila.find_element(By.XPATH, ".//td").text
                                if key and val:
                                    specs[key] = val
                            except:
                                continue

                        producto = {
                            "nombre": nombre,
                            "precio": precio,
                            "moneda": moneda,
                            "vendedor": vendedor,
                            "especificaciones": specs
                        }
                        resultados.append(producto)
                        print(f"✔️ Procesado: {nombre[:40]}...")

                    except Exception as e:
                        print(f"⚠️ Error en producto {i}: {e}")

                    time.sleep(random.uniform(15,20))

                    driver.execute_script("window.history.go(-1)")

                    time.sleep(random.uniform(3, 4))
                    driver.execute_script(f"window.scrollTo(0, {random.uniform(600, 800)});")

        finally:
            driver.get(page_link)
            time.sleep(random.uniform(10,15))
            
            try:
                siguiente = "Seguinte" if country == "Brasil" else "Siguiente"
                next_button = wait.until(
                        EC.presence_of_element_located((By.XPATH, 
                                                    f"//a[@class='andes-pagination__link' and @title='{siguiente}']"))
                        )
                
                driver.execute_script("arguments[0].scrollIntoView(true);", next_button)
                driver.save_screenshot("screenshot.png")

                print("Clickeando siguiente página: ", next_button.text)
                next_button.click()
                time.sleep(5)
                page_link = driver.current_url
                print("Página actual: ", page_link)
            
            except Exception as e :
                print("Llegamos a la página final")
                print(e)
                next_button_flag = 0

            curr_df = pd.DataFrame(resultados)
            curr_df.to_excel(f"{country}/df_{n_page}.xlsx", index=False)

            n_page += 1

    driver.quit()

    df = pd.DataFrame(resultados)
    df.to_excel("final_df.xlsx")
    return df


In [9]:
final_df = scrapear_mercadolibre("Paraguay", 1)

🏠 Entrando a la Home para 'calentar' la sesión...
🚀 Accediendo a: https://listado.mercadolibre.com.py/celulares
🔎 Buscando enlaces de productos...
Total de tarjetas detectadas: 48
🔍 Buscando específicamente clases que inicien con: poly-component__title
   📥 Scroll 1: Encontrados 48 nuevos (Total: 48)
   📥 Scroll 2: Encontrados 0 nuevos (Total: 48)
   📥 Scroll 3: Encontrados 0 nuevos (Total: 48)
   📥 Scroll 4: Encontrados 0 nuevos (Total: 48)
   📥 Scroll 5: Encontrados 0 nuevos (Total: 48)
   📥 Scroll 6: Encontrados 0 nuevos (Total: 48)
   📥 Scroll 7: Encontrados 0 nuevos (Total: 48)
   📥 Scroll 8: Encontrados 0 nuevos (Total: 48)
✅ Se encontraron 48 productos únicos.
--- [1/48] Navegando a producto ---
   ℹ️ El botón no estaba presente (quizás las specs ya están visibles).
Message: 
Stacktrace:
#0 0x5e0d1e19ba6a <unknown>
#1 0x5e0d1dbaaab5 <unknown>
#2 0x5e0d1dbfd676 <unknown>
#3 0x5e0d1dbfd8b1 <unknown>
#4 0x5e0d1dc48614 <unknown>
#5 0x5e0d1dc457b6 <unknown>
#6 0x5e0d1dbf0cbf <unknown